# Exploración computacional del Modelo 4

Este notebook documenta la implementación y validación experimental del **Modelo 4 — Costos detallados sala–sala**.

El objetivo es revisar, test por test, qué propiedad del modelo se está probando, qué datos se utilizan de forma resumida, cuál es el resultado esperado y cuál es el resultado obtenido al resolver con Gurobi.

En este modelo, los estudiantes que continúan desde un curso del bloque 1 hacia un curso del bloque 2 generan costo de movilidad. En cambio, los estudiantes que entran desde fuera al segundo bloque y los estudiantes que se van después del primer bloque no generan costo directo, aunque sí afectan las restricciones de capacidad.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Detectar raíz del repositorio.
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

BASE_DIR = ROOT_DIR / "data" / "modelo4"

from src.modelo_4_gurobi import resolver_modelo_4

In [2]:
def leer_csv(nombre_test, archivo):
    ruta = BASE_DIR / nombre_test / archivo
    return pd.read_csv(ruta)


def leer_metadata(nombre_test):
    ruta = BASE_DIR / nombre_test / "metadata.json"

    if not ruta.exists():
        return {}

    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)


def normalizar_tamano(df):
    df = df.copy()

    if "tamaño" in df.columns:
        df = df.rename(columns={"tamaño": "tamano"})

    return df


def mostrar_datos_test(nombre_test):
    cursos_b1 = normalizar_tamano(leer_csv(nombre_test, "cursos_b1.csv"))
    cursos_b2 = normalizar_tamano(leer_csv(nombre_test, "cursos_b2.csv"))
    salas = leer_csv(nombre_test, "salas.csv")
    flujos = leer_csv(nombre_test, "flujos.csv")
    libres = leer_csv(nombre_test, "libres.csv")
    entrantes = leer_csv(nombre_test, "entrantes.csv")
    costos = leer_csv(nombre_test, "costos_sala_sala.csv")
    metadata = leer_metadata(nombre_test)

    resumen = pd.DataFrame([
        {
            "Cursos B1": len(cursos_b1),
            "Cursos B2": len(cursos_b2),
            "Salas": len(salas),
            "Flujo entre bloques": flujos["flujo"].sum(),
            "Estudiantes salientes": libres["libres"].sum(),
            "Estudiantes entrantes": entrantes["entrantes"].sum(),
            "Status esperado": metadata.get("status_esperado"),
            "Costo esperado": metadata.get("costo_esperado"),
        }
    ])

    display(Markdown("**Resumen de la instancia**"))
    display(resumen)

    display(Markdown("**Cursos bloque 1**"))
    display(cursos_b1)

    display(Markdown("**Cursos bloque 2**"))
    display(cursos_b2)

    display(Markdown("**Salas**"))
    display(salas)

    display(Markdown("**Flujos entre bloques**"))
    display(flujos)

    display(Markdown("**Estudiantes salientes**"))
    display(libres)

    display(Markdown("**Estudiantes entrantes**"))
    display(entrantes)

    display(Markdown("**Matriz de costos sala–sala**"))
    try:
        matriz = costos.pivot(
            index="sala_origen",
            columns="sala_destino",
            values="costo"
        )
        display(matriz)
    except Exception:
        display(costos)


def resolver_y_mostrar_test(nombre_test):
    metadata = leer_metadata(nombre_test)
    status_esperado = metadata.get("status_esperado")
    costo_esperado = metadata.get("costo_esperado")

    try:
        resultado = resolver_modelo_4(
            BASE_DIR / nombre_test,
            mostrar_output=False,
            validar=True
        )

        status_obtenido = resultado["status"]
        costo_obtenido = resultado["objetivo"]

        if costo_esperado is None:
            coincide_costo = "No aplica"
        elif costo_obtenido is None:
            coincide_costo = "No aplica"
        else:
            coincide_costo = abs(float(costo_obtenido) - float(costo_esperado)) <= 1e-6

        resumen_resultado = pd.DataFrame([
            {
                "Status esperado": status_esperado,
                "Status obtenido": status_obtenido,
                "Costo esperado": costo_esperado,
                "Costo obtenido": costo_obtenido,
                "Coincide status": status_obtenido == status_esperado,
                "Coincide costo": coincide_costo,
            }
        ])

        display(Markdown("**Resultado obtenido**"))
        display(resumen_resultado)

        if resultado["status"] == "OPTIMAL":
            display(Markdown("**Asignación resultante por sala**"))
            display(resultado["df_salas"])

            display(Markdown("**Movimientos que generan costo**"))
            display(resultado["df_movimientos"])

        elif resultado["status"] == "INFEASIBLE":
            display(Markdown("La instancia fue detectada como **infactible** por Gurobi."))

        else:
            display(Markdown(f"Gurobi retornó status `{resultado['status']}`."))

    except Exception as e:
        status_obtenido = "ERROR_VALIDACION"

        resumen_error = pd.DataFrame([
            {
                "Status esperado": status_esperado,
                "Status obtenido": status_obtenido,
                "Costo esperado": costo_esperado,
                "Costo obtenido": None,
                "Coincide status": status_obtenido == status_esperado,
                "Mensaje": str(e),
            }
        ])

        display(Markdown("**Resultado obtenido**"))
        display(resumen_error)

## M4_T01 — Misma sala, costo 0


Este test verifica que, cuando los cursos del bloque 2 pueden ser asignados a la misma sala que los cursos del bloque 1 que les envían estudiantes, el costo total de movilidad es cero.

Se prueba la propiedad básica del modelo: permanecer en la misma sala no genera costo.


Resultado esperado: `OPTIMAL`, con costo óptimo `0`.

In [3]:
mostrar_datos_test("M4_T01_misma_sala_costo_0")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,2,50,0,0,OPTIMAL,0


**Cursos bloque 1**

,curso_id,tamano
0,i1,30
1,i2,20


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2
sala_origen,,
r1,0,5
r2,5,0


In [4]:
resolver_y_mostrar_test("M4_T01_misma_sala_costo_0")

Set parameter Username
Set parameter LicenseID to value 2808412
Academic license - for non-commercial use only - expires 2027-04-16


**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,0,0.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,30,A,i1,30,k1,30
1,r2,20,B,i2,20,k2,20


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r1,k1,r1,30.0,0.0,0.0
1,i2,r2,k2,r2,20.0,0.0,0.0


## M4_T02 — Costo detallado forzado por capacidad


Este test verifica que el Modelo 4 utiliza el costo detallado sala–sala y no simplemente un costo binario.

Las capacidades fuerzan una asignación donde algunos estudiantes deben cambiar de sala. Como el costo entre salas es 2, el costo esperado aumenta respecto de un modelo binario.


Resultado esperado: `OPTIMAL`, con costo óptimo `120`.

In [5]:
mostrar_datos_test("M4_T02_costo_detallado_forzado")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,2,80,0,0,OPTIMAL,120


**Cursos bloque 1**

,curso_id,tamano
0,i1,50
1,i2,30


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2
sala_origen,,
r1,0,2
r2,2,0


In [6]:
resolver_y_mostrar_test("M4_T02_costo_detallado_forzado")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,120,120.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,50,A,i1,50,k2,50
1,r2,30,A,i2,30,k1,30


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r1,k1,r2,30.0,2.0,60.0
1,i1,r1,k2,r1,20.0,0.0,0.0
2,i2,r2,k2,r1,30.0,2.0,60.0


## M4_T03 — Elección de salas por matriz de costos


Este test verifica que, cuando existen más salas que cursos, el modelo escoge las salas que minimizan el costo según la matriz detallada.

La matriz de costos hace que ciertas salas sean más convenientes que otras, aun cuando todas tengan capacidad suficiente.


Resultado esperado: `OPTIMAL`, con costo óptimo `8`.

In [7]:
mostrar_datos_test("M4_T03_eleccion_por_matriz_costos")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,3,20,0,0,OPTIMAL,8


**Cursos bloque 1**

,curso_id,tamano
0,i1,10
1,i2,10


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0,1,5
r2,1,0,5
r3,5,5,0


In [8]:
resolver_y_mostrar_test("M4_T03_eleccion_por_matriz_costos")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,8,8.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,10,A,i2,10,k2,10
1,r2,10,A,i1,10,k1,10
2,r3,10,B,—,—,—,—


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r2,k1,r2,6.0,0.0,0.0
1,i1,r2,k2,r1,4.0,1.0,4.0
2,i2,r1,k1,r2,4.0,1.0,4.0
3,i2,r1,k2,r1,6.0,0.0,0.0


## M4_T04 — Estudiantes entrantes al segundo bloque


Este test incorpora estudiantes que llegan desde fuera al segundo bloque.

Estos estudiantes aumentan el tamaño del curso del bloque 2 y, por lo tanto, afectan la restricción de capacidad. Sin embargo, no generan costo de movilidad porque no tienen sala de origen en el bloque 1.


Resultado esperado: `OPTIMAL`, con costo óptimo `0`.

In [9]:
mostrar_datos_test("M4_T04_estudiantes_entrantes")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,3,50,0,20,OPTIMAL,0


**Cursos bloque 1**

,curso_id,tamano
0,i1,30
1,i2,20


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,20
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0,2,4
r2,2,0,1
r3,4,1,0


In [10]:
resolver_y_mostrar_test("M4_T04_estudiantes_entrantes")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,0,0.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,50,A,i1,30,k1,50
1,r2,30,A,i2,20,k2,20
2,r3,20,B,—,—,—,—


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r1,k1,r1,30.0,0.0,0.0
1,i2,r2,k2,r2,20.0,0.0,0.0


## M4_T05 — Estudiantes salientes desde el primer bloque


Este test incorpora estudiantes que tuvieron clase en el bloque 1, pero no continúan al bloque 2.

Estos estudiantes afectan la capacidad requerida en el bloque 1, pero no generan costo de movilidad porque no tienen sala de destino en el segundo bloque.


Resultado esperado: `OPTIMAL`, con costo óptimo `0`.

In [11]:
mostrar_datos_test("M4_T05_estudiantes_salientes")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,3,50,20,0,OPTIMAL,0


**Cursos bloque 1**

,curso_id,tamano
0,i1,50
1,i2,20


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,20
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0,2,4
r2,2,0,1
r3,4,1,0


In [12]:
resolver_y_mostrar_test("M4_T05_estudiantes_salientes")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,0,0.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,50,A,i1,50,k1,30
1,r2,30,A,i2,20,k2,20
2,r3,20,B,—,—,—,—


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r1,k1,r1,30.0,0.0,0.0
1,i2,r2,k2,r2,20.0,0.0,0.0


## M4_T06 — Entrantes y salientes cambian el óptimo por capacidad


Este test combina estudiantes entrantes y salientes.

La idea es verificar que estos estudiantes no aparecen directamente en la función objetivo, pero sí pueden modificar la solución óptima al cambiar los tamaños efectivos de los cursos y activar restricciones de capacidad.


Resultado esperado: `OPTIMAL`, con costo óptimo `120`.

In [13]:
mostrar_datos_test("M4_T06_entrantes_salientes_cambian_optimo")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,2,60,20,20,OPTIMAL,120


**Cursos bloque 1**

,curso_id,tamano
0,i1,50
1,i2,30


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,20
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,20


**Matriz de costos sala–sala**

sala_destino,r1,r2
sala_origen,,
r1,0,2
r2,2,0


In [14]:
resolver_y_mostrar_test("M4_T06_entrantes_salientes_cambian_optimo")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,120,120.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,50,A,i1,50,k2,50
1,r2,30,A,i2,30,k1,30


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r1,k1,r2,30.0,2.0,60.0
1,i2,r2,k2,r1,30.0,2.0,60.0


## M4_T07 — Infactible por estudiantes entrantes


Este test verifica que el modelo detecta infactibilidad cuando los estudiantes entrantes hacen que un curso del bloque 2 no quepa en ninguna sala.

Aunque solo una parte de los estudiantes proviene del bloque 1, la capacidad debe cubrir el tamaño total del curso del bloque 2.


Resultado esperado: `INFEASIBLE`.

In [15]:
mostrar_datos_test("M4_T07_infactible_por_entrantes")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,1,1,2,30,0,30,INFEASIBLE,None


**Cursos bloque 1**

,curso_id,tamano
0,i1,30


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,30


**Matriz de costos sala–sala**

sala_destino,r1,r2
sala_origen,,
r1,0,3
r2,3,0


In [16]:
resolver_y_mostrar_test("M4_T07_infactible_por_entrantes")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,INFEASIBLE,INFEASIBLE,None,None,True,No aplica


La instancia fue detectada como **infactible** por Gurobi.

## M4_T08 — Infactible por estudiantes salientes


Este test verifica que el modelo detecta infactibilidad cuando un curso del bloque 1 no cabe en ninguna sala.

Aunque parte de sus estudiantes se vaya después del primer bloque, todos deben caber en la sala asignada durante el bloque 1.


Resultado esperado: `INFEASIBLE`.

In [17]:
mostrar_datos_test("M4_T08_infactible_por_salientes")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,1,1,2,30,30,0,INFEASIBLE,None


**Cursos bloque 1**

,curso_id,tamano
0,i1,60


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,30


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0


**Matriz de costos sala–sala**

sala_destino,r1,r2
sala_origen,,
r1,0,3
r2,3,0


In [18]:
resolver_y_mostrar_test("M4_T08_infactible_por_salientes")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,INFEASIBLE,INFEASIBLE,None,None,True,No aplica


La instancia fue detectada como **infactible** por Gurobi.

## M4_T09 — Matriz de costos incompleta


Este test no está diseñado para ser resuelto por Gurobi.

Su objetivo es verificar que la validación detecte una matriz de costos sala–sala incompleta. En el Modelo 4 debe existir un costo para cada par ordenado de salas.


Resultado esperado: `ERROR_VALIDACION`.

In [19]:
mostrar_datos_test("M4_T09_matriz_costos_incompleta")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,3,20,0,0,ERROR_VALIDACION,None


**Cursos bloque 1**

,curso_id,tamano
0,i1,10
1,i2,10


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0.0,1.0,5.0
r2,1.0,0.0,5.0
r3,NaN,5.0,0.0


In [20]:
resolver_y_mostrar_test("M4_T09_matriz_costos_incompleta")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Mensaje
0,ERROR_VALIDACION,ERROR_VALIDACION,None,None,True,La matriz de costos está incompleta. Faltan pa...


## M4_T10 — Costos no equivalentes a gamma por edificio


Este test verifica la diferencia conceptual entre el Modelo 3 y el Modelo 4.

En el Modelo 3, el costo depende del tipo de transición o del edificio. En el Modelo 4, el costo depende directamente del par de salas. Por eso, una sala en otro edificio puede ser más barata si así lo indica la matriz detallada.


Resultado esperado: `OPTIMAL`, con costo óptimo `16`.

In [21]:
mostrar_datos_test("M4_T10_costos_no_equivalentes_a_gamma")

**Resumen de la instancia**

,Cursos B1,Cursos B2,Salas,Flujo entre bloques,Estudiantes salientes,Estudiantes entrantes,Status esperado,Costo esperado
0,2,2,3,20,0,0,OPTIMAL,16


**Cursos bloque 1**

,curso_id,tamano
0,i1,10
1,i2,10


**Cursos bloque 2**

...

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes**

,curso_b2,entrantes
0,k1,0
1,k2,0


**Matriz de costos sala–sala**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0,10,2
r2,10,0,10
r3,2,10,0


In [22]:
resolver_y_mostrar_test("M4_T10_costos_no_equivalentes_a_gamma")

**Resultado obtenido**

,Status esperado,Status obtenido,Costo esperado,Costo obtenido,Coincide status,Coincide costo
0,OPTIMAL,OPTIMAL,16,16.0,True,True


**Asignación resultante por sala**

,sala,capacidad,edificio,curso_b1,tamano_b1,curso_b2,tamano_b2
0,r1,10,A,i1,10,k1,10
1,r2,10,A,—,—,—,—
2,r3,10,B,i2,10,k2,10


**Movimientos que generan costo**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total
0,i1,r1,k1,r1,6.0,0.0,0.0
1,i1,r1,k2,r3,4.0,2.0,8.0
2,i2,r3,k1,r1,4.0,2.0,8.0
3,i2,r3,k2,r3,6.0,0.0,0.0


## Síntesis general

Los tests anteriores permiten verificar separadamente las propiedades centrales del Modelo 4:

- uso de una matriz completa de costos sala–sala;
- efecto de capacidades heterogéneas;
- incorporación de estudiantes entrantes al segundo bloque;
- incorporación de estudiantes salientes desde el primer bloque;
- detección de infactibilidad por capacidad;
- detección de errores estructurales en la matriz de costos;
- diferencia conceptual respecto del Modelo 3, donde el costo dependía principalmente del tipo de transición entre edificios.

En conjunto, estos casos permiten comprobar que los estudiantes entrantes y salientes no afectan directamente la función objetivo, pero sí pueden modificar la factibilidad o la solución óptima a través de las restricciones de capacidad.